<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# LogiTrack — Supply Chain Analytics
## Notebook 5 — Machine Learning Premium : Prédiction des retards
### ✅ VERSION CORRIGÉE

> **Comment lire ce corrigé :**  
> Les blocs `MÉTHODE` expliquent les choix techniques et les patterns généralisables.  
> Les blocs `INTERPRÉTATION` lisent les résultats.  
> Les blocs `MÉTIER` font le lien entre le chiffre et la décision business.

| | |
|---|---|
| **Prérequis** | `logitrack_analytics.csv` dans `SAVE_PATH` (produit par NB2) |
| **Niveau** | Premium |
| **Outils** | Python — scikit-learn, pandas, matplotlib |
| **Durée estimée** | 3h à 4h |

> **Problème :** Classification binaire — prédire `livraison_en_retard = 1` avant l'expédition.
>
> **Contrainte métier :** Recall > 0.75. Il vaut mieux alerter par excès (fausse alarme) que rater un vrai retard. Le coût d'une fausse alerte (dispatcher qui vérifie un colis qui arrivera à temps) est bien inférieur au coût d'un retard non détecté (pénalité + client mécontent).
>
> **Coupure temporelle :** Train avant 2023-07-01 / Test après 2023-07-01.

---
## Étape 1 — Imports et configuration

> **MÉTHODE — Pourquoi `TimeSeriesSplit` et non `KFold` ?**
>
> Les données de livraison sont temporelles — une livraison créée en janvier 2023 ne peut pas être utilisée pour prédire une livraison de décembre 2022. `KFold` ignore l'ordre temporel et crée du **leakage** : le modèle apprend sur le futur pour prédire le passé. `TimeSeriesSplit` garantit que chaque fold de test est toujours plus récent que le fold d'entraînement.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
    'light':     '#EEEDFE',
}
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/logitrack/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

---
## Étape 2 — Chargement et vérification

### MÉTHODE
On vérifie la distribution de la variable cible et les nulls dans les features ML avant toute modélisation. Un dataset déséquilibré (< 20% de classe positive) nécessitera `class_weight='balanced'` ou SMOTE. Les nulls résiduels seront imputés par la médiane — stratégie robuste pour des features numériques.

In [ ]:
BASE_URL   = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/logitrack/data/'
clean_path = f'{SAVE_PATH}logitrack_analytics.csv'

if not os.path.exists(clean_path):
    clean_path = BASE_URL + 'logitrack_analytics.csv'
    print('⚠️  logitrack_analytics.csv non trouvé en local — chargement depuis GitHub')
else:
    print(f'✅ Fichier trouvé : {clean_path}')

df = pd.read_csv(clean_path, parse_dates=['date_creation','date_enlevement','date_livraison_prevue'])

# Exclure transporteur suspendu
df = df[df['transporteur_actif'] == 1].copy()
print(f'Dataset : {df.shape}')

# Distribution variable cible
print(f'\nDistribution livraison_en_retard :')
vc = df['livraison_en_retard'].value_counts()
print(vc)
print(f'  Taux classe positive : {vc[1]/len(df)*100:.1f}%')

# Features ML disponibles
FEATURES = [
    # Temporel
    'mois_sin', 'mois_cos', 'jour_semaine', 'est_weekend',
    'est_fin_mois', 'heure_enlevement', 'delai_prep_heures',
    # Logistique
    'poids_kg', 'volume_m3', 'densite_kg_m3',
    'priorite_num', 'sla_jours_contractuel',
    'distance_km', 'duree_standard_jours', 'nb_points_controle',
    # Performance historique
    'taux_breach_transporteur', 'taux_breach_corridor',
    'taux_breach_entrepot_origine', 'taux_livraison_temps',
    'ponctualite_entrepot_origine',
    # Contexte
    'risque_douanier_num', 'mode_transport_num',
    'segment_num', 'penalite_retard_pct',
]

print(f'\nNulls dans les features :')
nulls = df[FEATURES].isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else '  Aucun null ✅')

> **INTERPRÉTATION :**
> - Distribution quasi-équilibrée (~39% positifs) — pas besoin de SMOTE
> - 24 features ML couvrant 5 dimensions : temporel, logistique, performance historique, géographie, contexte client
> - Le `transporteur_actif = 0` (TRP15) est exclu — ses patterns de retard ne doivent pas biaiser l'entraînement
>
> **MÉTIER :** Les features de **performance historique** (`taux_breach_transporteur`, `taux_breach_corridor`) sont particulièrement importantes — elles encodent la connaissance opérationnelle que les dispatchers ont déjà en tête sans la formaliser.

---
## Étape 3 — Coupure temporelle

### MÉTHODE
La coupure temporelle est **la règle la plus importante** du ML sur données de flux. En utilisant `date_creation < 2023-07-01` pour le train et `>= 2023-07-01` pour le test, on simule les conditions réelles de production : le modèle est entraîné sur le passé et prédit sur le futur.

**Features exclues pour éviter le data leakage :**
- `delai_jours` — contient le résultat final de la livraison
- `sla_breach`, `escalade`, `in_backlog` — composantes de la variable cible
- `date_livraison_reelle` — inconnue au moment de l'expédition
- `duree_reelle_jours` — inconnue au moment de l'expédition
- `ratio_duree_vs_standard` — calculé depuis `duree_reelle_jours`
- `delai_relatif_sla` — calculé depuis `duree_reelle_jours`

In [ ]:
CIBLE        = 'livraison_en_retard'
DATE_COUPURE = pd.Timestamp('2023-07-01')

df_ml = df[FEATURES + [CIBLE, 'date_creation']].copy()
# Imputation médiane pour les nulls résiduels
for col in FEATURES:
    if df_ml[col].isnull().sum() > 0:
        df_ml[col] = df_ml[col].fillna(df_ml[col].median())

train = df_ml[df_ml['date_creation'] < DATE_COUPURE]
test  = df_ml[df_ml['date_creation'] >= DATE_COUPURE]
X_train, y_train = train[FEATURES], train[CIBLE]
X_test,  y_test  = test[FEATURES],  test[CIBLE]

print(f'Train : {len(train):,} lignes | taux positifs : {y_train.mean()*100:.1f}%')
print(f'Test  : {len(test):,} lignes  | taux positifs : {y_test.mean()*100:.1f}%')
print(f'Ratio train/test : {len(train)/len(df_ml)*100:.0f}/{len(test)/len(df_ml)*100:.0f}%')

> **INTERPRÉTATION :**
> - **Train : ~75%** des livraisons (janv. 2022 — juin 2023)
> - **Test : ~25%** des livraisons (juil. 2023 — déc. 2023)
> - Le taux de positifs peut différer entre train et test selon la saisonnalité — c'est normal et attendu
>
> **MÉTIER :** En production, le modèle sera ré-entraîné chaque trimestre avec les nouvelles livraisons. La coupure glissante garantit que le modèle s'adapte aux évolutions du réseau (nouveaux transporteurs, nouvelles routes, changements de SLA).

---
## Étape 4 — Entraînement de 3 modèles

### MÉTHODE
`class_weight='balanced'` pénalise davantage les erreurs sur la classe minoritaire en pondérant inversement les classes par leur fréquence. Même avec un dataset quasi-équilibré (39/61), cela améliore le Recall car le modèle est incité à ne pas ignorer les cas positifs.

**Standardisation :** `StandardScaler` uniquement pour la Logistic Regression — les arbres de décision (RF, GB) sont insensibles à l'échelle des features.

In [ ]:
# Standardisation pour Logistic Regression uniquement
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr  = lr.predict(X_test_sc)
y_proba_lr = lr.predict_proba(X_test_sc)[:, 1]

# Random Forest
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

# Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=200, max_depth=4,
    learning_rate=0.05, random_state=42
)
gb.fit(X_train, y_train)
y_pred_gb  = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

# Tableau comparatif
results = pd.DataFrame({
    'Modèle':    ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'AUC-ROC':   [roc_auc_score(y_test, y_proba_lr),
                  roc_auc_score(y_test, y_proba_rf),
                  roc_auc_score(y_test, y_proba_gb)],
    'F1':        [f1_score(y_test, y_pred_lr),
                  f1_score(y_test, y_pred_rf),
                  f1_score(y_test, y_pred_gb)],
    'Recall':    [recall_score(y_test, y_pred_lr),
                  recall_score(y_test, y_pred_rf),
                  recall_score(y_test, y_pred_gb)],
    'Precision': [precision_score(y_test, y_pred_lr),
                  precision_score(y_test, y_pred_rf),
                  precision_score(y_test, y_pred_gb)],
}).round(4)
print(results.to_string(index=False))

> **INTERPRÉTATION — Comparaison des 3 modèles (seuil par défaut 0.5) :**
>
> - **Logistic Regression** : AUC modéré mais Recall élevé grâce à `class_weight='balanced'`
> - **Random Forest** : meilleur AUC-ROC (~0.85+) — distingue bien les classes mais Recall faible au seuil par défaut
> - **Gradient Boosting** : AUC similaire au RF avec une meilleure précision
>
> **Conclusion :** Le Random Forest a le meilleur AUC — il distingue bien les retards des livraisons à temps sur l'ensemble des seuils. Il suffit d'abaisser son seuil de décision pour atteindre Recall > 0.75.

---
## Étape 5 — Optimisation du seuil de décision

### MÉTHODE
L'AUC-ROC mesure la capacité discriminante du modèle sur **tous** les seuils possibles. Un AUC de 0.85 signifie que le modèle classe correctement 85% des paires (retard, à temps). Le seuil par défaut (0.5) n'est pas toujours optimal pour une contrainte métier spécifique.

**Principe :** abaisser le seuil → plus d'alertes → Recall augmente, Precision baisse. On cherche le seuil le plus élevé satisfaisant Recall ≥ 0.75 — pour minimiser les fausses alertes tout en couvrant les vrais retards.

In [ ]:
seuils = np.arange(0.25, 0.70, 0.05)
rows = []
for s in seuils:
    yp = (y_proba_rf >= s).astype(int)
    rows.append({
        'seuil':     round(s, 2),
        'recall':    recall_score(y_test, yp),
        'precision': precision_score(y_test, yp, zero_division=0),
        'f1':        f1_score(y_test, yp, zero_division=0),
        'nb_alertes': int(yp.sum()),
        'pct_alertes': round(yp.mean()*100, 1)
    })
df_s = pd.DataFrame(rows)
print(df_s.round(4).to_string(index=False))

# Premier seuil satisfaisant Recall >= 0.75
optimal = df_s[df_s['recall'] >= 0.75].iloc[0]
SEUIL_OPTIMAL = float(optimal['seuil'])
print(f'\nSeuil optimal retenu : {SEUIL_OPTIMAL:.2f}')
print(f'  Recall    : {optimal["recall"]:.4f}')
print(f'  Precision : {optimal["precision"]:.4f}')
print(f'  F1-Score  : {optimal["f1"]:.4f}')
print(f'  Nb alertes: {optimal["nb_alertes"]:.0f} ({optimal["pct_alertes"]}% du test set)')

> **INTERPRÉTATION — Tableau des seuils :**
>
> | Seuil | Recall | Precision | Commentaire |
> |---|---|---|---|
> | 0.25 | ~0.95+ | ~0.55 | Trop d'alertes — équipe submergée |
> | 0.35 | ~0.85 | ~0.65 | Bon compromis |
> | 0.40 | ~0.78 | ~0.70 | **Seuil optimal** — Recall ≥ 0.75 + Precision max |
> | 0.50 | ~0.55 | ~0.80 | Seuil défaut — Recall insuffisant |
>
> **MÉTIER :** Avec le seuil optimal, pour 100 vrais retards le modèle en détecte ~78 et génère ~30% de fausses alertes. En pratique : un dispatcher traite en priorité les 20 alertes les plus critiques (score > 0.70) — les autres sont traitées au second plan. C'est exactement ce que demandait le Directeur des Opérations.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(df_s['seuil'], df_s['recall'],    color=COLORS['danger'],    label='Recall',
        linewidth=2.5, marker='o')
ax.plot(df_s['seuil'], df_s['precision'], color=COLORS['secondary'], label='Precision',
        linewidth=2.5, marker='s')
ax.plot(df_s['seuil'], df_s['f1'],        color=COLORS['primary'],   label='F1-Score',
        linewidth=2, linestyle='--')
ax.axvline(SEUIL_OPTIMAL, color=COLORS['warning'], linestyle='--', linewidth=2,
           label=f'Seuil optimal ({SEUIL_OPTIMAL:.2f})')
ax.axhline(0.75, color=COLORS['danger'], linestyle=':', linewidth=1, alpha=0.7,
           label='Contrainte Recall = 0.75')
ax.set_xlabel('Seuil de décision')
ax.set_ylabel('Score')
ax.set_title('Precision vs Recall selon le seuil — Random Forest LogiTrack',
             fontweight='bold')
ax.legend()
ax.set_xlim(0.23, 0.72)
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}logitrack_precision_recall_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}logitrack_precision_recall_curve.png')

---
## Étape 6 — Feature Importance

### MÉTHODE
La feature importance du Random Forest mesure la réduction moyenne d'impureté (Gini) apportée par chaque feature lors des splits. Une feature importante est celle qui discrimine le mieux les livraisons en retard des livraisons à temps. C'est aussi un outil de communication — elle dit aux opérationnels sur quels indicateurs se concentrer.

In [ ]:
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('=== FEATURE IMPORTANCE ===')
for feat, imp in fi.items():
    bar = '█' * int(imp * 300)
    print(f'  {feat:<35} {imp:.4f} ({imp*100:.1f}%) {bar}')

> **INTERPRÉTATION — Features les plus importantes :**
>
> | Rang | Feature | Importance | Interprétation |
> |---|---|---|---|
> | 1 | `taux_breach_transporteur` | **~22%** | Le transporteur est le prédicteur le plus fort |
> | 2 | `taux_breach_corridor` | **~18%** | La route concentre le risque structurel |
> | 3 | `risque_douanier_num` | **~14%** | Élevé vs Faible = différence majeure |
> | 4 | `nb_points_controle` | **~10%** | Plus de frontières = plus de risque |
> | 5 | `ponctualite_entrepot_origine` | **~9%** | Le retard commence à l'entrepôt |
> | 6 | `delai_prep_heures` | **~8%** | Temps de préparation révèle les problèmes amont |
> | 7 | `taux_livraison_temps` | **~7%** | Fiabilité historique du transporteur |
>
> **MÉTIER :** Les 4 features les plus importantes sont toutes liées au **transporteur** et au **corridor** — pas au type de colis, pas au client, pas à la météo. La décision de confier une livraison à TRP12 sur un corridor sahélien est la combinaison la plus prédictive d'un retard. Cette décision est prise avant l'expédition — le modèle peut alerter dès la création du bon de livraison.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
fi_sorted   = fi.sort_values()
colors_fi   = [
    COLORS['danger']  if imp > 0.18 else
    COLORS['warning'] if imp > 0.10 else
    COLORS['primary'] if imp > 0.05 else
    COLORS['neutral']
    for imp in fi_sorted.values
]
ax.barh(fi_sorted.index, fi_sorted.values, color=colors_fi)
for i, (feat, imp) in enumerate(fi_sorted.items()):
    ax.text(imp + 0.002, i, f'{imp*100:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Importance (%)')
ax.set_title('Feature Importance — Random Forest LogiTrack', fontweight='bold')
ax.set_xlim(0, fi.max() * 1.3)
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}logitrack_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Sauvegardé : {SAVE_PATH}logitrack_feature_importance.png')

---
## Étape 7 — Validation par TimeSeriesSplit

### MÉTHODE
`TimeSeriesSplit(n_splits=5)` crée 5 folds en respectant l'ordre temporel. Un modèle stable doit avoir un écart-type de Recall < 0.10 entre les folds. Un écart-type élevé signale que le modèle est sensible à la période — il faudra alors investiguer si des changements opérationnels (nouveau transporteur, nouvelle route) ont cassé la stationnarité des patterns.

In [ ]:
df_sorted = df_ml.sort_values('date_creation').reset_index(drop=True)
X_all     = df_sorted[FEATURES].fillna(df_sorted[FEATURES].median())
y_all     = df_sorted[CIBLE]

tscv       = TimeSeriesSplit(n_splits=5)
recalls_cv = []
aucs_cv    = []

for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_all)):
    rf_cv = RandomForestClassifier(
        n_estimators=100, max_depth=8,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    rf_cv.fit(X_all.iloc[tr_idx], y_all.iloc[tr_idx])
    y_proba_cv = rf_cv.predict_proba(X_all.iloc[te_idx])[:, 1]
    y_pred_cv  = (y_proba_cv >= SEUIL_OPTIMAL).astype(int)
    r  = recall_score(y_all.iloc[te_idx], y_pred_cv)
    au = roc_auc_score(y_all.iloc[te_idx], y_proba_cv)
    recalls_cv.append(r)
    aucs_cv.append(au)
    print(f'  Fold {fold+1} | Train: {len(tr_idx):>5,} | Test: {len(te_idx):>5,} '
          f'| Recall: {r:.4f} | AUC: {au:.4f}')

print(f'\nRecall moyen  : {np.mean(recalls_cv):.4f}')
print(f'AUC moyen     : {np.mean(aucs_cv):.4f}')
print(f'Écart-type RC : {np.std(recalls_cv):.4f}')
print(f'Stable        : {"✅ Oui" if np.std(recalls_cv) < 0.10 else "⚠️ Non — instabilité détectée"}')

> **INTERPRÉTATION — TimeSeriesSplit :**
>
> - Le Fold 1 (peu de données d'entraînement) a généralement un Recall plus faible
> - Les Folds 2-5 convergent vers un Recall stable autour de 0.75-0.85
> - Un écart-type < 0.10 confirme que le modèle généralise bien dans le temps
>
> **MÉTIER :** En production, surveiller le Recall sur les nouvelles données chaque mois est essentiel. Si le Recall chute sous 0.70 deux mois de suite, c'est le signal qu'il faut ré-entraîner le modèle — peut-être parce qu'un nouveau transporteur est entré en service ou qu'une route a été modifiée.

---
## Étape 8 — Génération du fichier d'alertes

### MÉTHODE
On applique le modèle sur **toutes les livraisons En transit** — ce sont les seules pour lesquelles une intervention est encore possible. Les livraisons déjà livrées ou perdues ne peuvent pas être sauvées.

Le fichier `logitrack_risque_scores.csv` est stratifié en **3 niveaux d'intervention** selon le score de risque — pour permettre aux dispatchers de prioriser leurs actions sans être submergés.

In [ ]:
# Appliquer le modèle sur les livraisons En transit
df_transit = df[df['statut'] == 'En transit'].copy()

if len(df_transit) == 0:
    # Fallback : livraisons récentes sans date de livraison réelle
    df_transit = df[df['date_livraison_reelle'].isna() &
                    df['statut'].isin(['En transit'])].copy()
    print(f'⚠️  Pas de livraisons En transit — utilisation du dataset complet pour la démo')
    df_transit = df.tail(500).copy()  # démo sur les 500 dernières livraisons

X_score = df_transit[FEATURES].fillna(df_transit[FEATURES].median())
scores  = rf.predict_proba(X_score)[:, 1]

df_transit = df_transit.copy()
df_transit['score_risque']     = scores
df_transit['alerte_retard']    = (scores >= SEUIL_OPTIMAL).astype(int)

# Stratification en 3 niveaux
def niveau_urgence(score):
    if score >= 0.75:   return 'Critique'
    elif score >= 0.55: return 'Élevé'
    elif score >= SEUIL_OPTIMAL: return 'Modéré'
    else:               return 'Faible'

df_transit['niveau_urgence'] = df_transit['score_risque'].apply(niveau_urgence)

# Colonnes utiles pour le dashboard
cols_export = [
    'livraison_id', 'client_id', 'transporteur_id',
    'pays_origine', 'pays_destination',
    'date_creation', 'date_livraison_prevue',
    'priorite', 'type_colis', 'poids_kg',
    'sla_jours_contractuel', 'risque_douanier',
    'taux_breach_transporteur', 'taux_breach_corridor',
    'score_risque', 'alerte_retard', 'niveau_urgence'
]
cols_export = [c for c in cols_export if c in df_transit.columns]
df_alertes = (
    df_transit[cols_export]
    .sort_values('score_risque', ascending=False)
    .reset_index(drop=True)
)

df_alertes.to_csv(f'{SAVE_PATH}logitrack_risque_scores.csv', index=False)
print(f'✅ logitrack_risque_scores.csv généré')
print(f'   Livraisons analysées : {len(df_alertes):,}')
print(f'   Alertes déclenchées  : {df_alertes["alerte_retard"].sum():,} '
      f'({df_alertes["alerte_retard"].mean()*100:.1f}%)')
print(f'   Dont Critique        : {(df_alertes["niveau_urgence"]=="Critique").sum():,}')
print(f'   Dont Élevé           : {(df_alertes["niveau_urgence"]=="Élevé").sum():,}')
print(f'   Dont Modéré          : {(df_alertes["niveau_urgence"]=="Modéré").sum():,}')
print(f'\nTop 5 alertes les plus urgentes :')
cols_show = ['livraison_id','pays_origine','pays_destination',
             'priorite','score_risque','niveau_urgence']
cols_show = [c for c in cols_show if c in df_alertes.columns]
print(df_alertes[cols_show].head(5).to_string())

> **INTERPRÉTATION :**
>
> **MÉTIER — Stratification des alertes en 3 niveaux :**
>
> | Score | Niveau | Action recommandée |
> |---|---|---|
> | > 0.75 | **Critique** | Appel téléphonique au transporteur + client prévenu |
> | 0.55-0.75 | **Élevé** | Email de suivi au transporteur + monitoring renforcé |
> | Seuil-0.55 | **Modéré** | Notification push dispatcher + vérification J+1 |
>
> Cette stratification permet de prioriser les interventions sans submerger l'équipe. Les alertes Critiques (score > 0.75) correspondent aux livraisons que le Directeur des Opérations voulait voir sur son dashboard chaque matin.

---
## Bilan du Notebook 5

| Metric | Logistic Regression | Random Forest | Gradient Boosting |
|---|---|---|---|
| AUC-ROC | ~0.72 | **~0.86** | ~0.85 |
| Recall (seuil déf.) | ~0.82 | ~0.52 | ~0.54 |

| Élément | Valeur |
|---|---|
| Modèle choisi | **Random Forest** (meilleur AUC) |
| Seuil optimal | Variable selon le dataset — satisfait Recall ≥ 0.75 |
| Feature #1 | `taux_breach_transporteur` (~22%) |
| Feature #2 | `taux_breach_corridor` (~18%) |
| Feature #3 | `risque_douanier_num` (~14%) |
| Validation CV | `TimeSeriesSplit` 5 folds — Recall stable ✅ |

**Fichiers exportés dans `SAVE_PATH` :**
- `logitrack_risque_scores.csv` — alertes stratifiées pour Power BI
- `logitrack_precision_recall_curve.png`
- `logitrack_feature_importance.png`

**Recommandation finale :** En production, ré-entraîner le modèle chaque trimestre et surveiller le Recall mensuel. Si Recall < 0.70 deux mois consécutifs → investigation des changements opérationnels récents (nouveau transporteur, nouvelle route, modification SLA).

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.